In [1]:
import requests
import pandas as pd
import time
import base64
import random
from datetime import datetime
from collections import Counter

random.seed(42)

In [ ]:
GITHUB_TOKEN = "GithubTokenHere"  # Replace with your GitHub token
OUTPUT_FILE  = "github_projects_data.xlsx"
DELAY        = 1.5    # seconds between repo fetches
PAGE_DELAY   = 3      # seconds between pages in same query
QUERY_DELAY  = 5      # seconds between different queries
MAX_PAGES    = 3      # pages per query (100 results each = up to 300 per query)

HEADERS = {
    "Authorization": f"token {GITHUB_TOKEN}",
    "Accept": "application/vnd.github.v3+json"
}

In [4]:
# ── SEARCH QUERIES (75 total across 10 domains) ────────────────────────────────
SEARCH_QUERIES = [
    # Ecommerce
    ("topic:ecommerce stars:>50 language:javascript",        "Ecommerce"),
    ("topic:ecommerce stars:>50 language:python",            "Ecommerce"),
    ("topic:ecommerce stars:>30 language:php",               "Ecommerce"),
    ("topic:ecommerce stars:>30 language:java",              "Ecommerce"),
    ("topic:shopping-cart stars:>30",                        "Ecommerce"),
    ("ecommerce platform fullstack stars:>100",              "Ecommerce"),
    ("topic:woocommerce stars:>40",                          "Ecommerce"),
    ("online marketplace fullstack stars:>50",               "Ecommerce"),
    ("topic:shopify stars:>30",                              "Ecommerce"),
    ("multi vendor ecommerce stars:>30",                     "Ecommerce"),
    # Healthcare
    ("topic:healthcare stars:>30",                           "Healthcare"),
    ("hospital management system stars:>20",                 "Healthcare"),
    ("topic:medical-records stars:>20",                      "Healthcare"),
    ("topic:telemedicine stars:>20",                         "Healthcare"),
    ("patient management system stars:>20",                  "Healthcare"),
    ("topic:ehr stars:>20",                                  "Healthcare"),
    ("clinic management system stars:>15",                   "Healthcare"),
    ("healthcare app fullstack stars:>20",                   "Healthcare"),
    ("pharmacy management system stars:>15",                 "Healthcare"),
    ("topic:health-app stars:>20",                           "Healthcare"),
    # Education
    ("topic:lms stars:>50",                                  "Education"),
    ("topic:elearning stars:>30",                            "Education"),
    ("online learning management system stars:>50",          "Education"),
    ("topic:online-course stars:>30",                        "Education"),
    ("topic:moodle stars:>30",                               "Education"),
    ("student management system stars:>20",                  "Education"),
    ("topic:quiz-app stars:>30",                             "Education"),
    ("virtual classroom stars:>20",                          "Education"),
    ("school management system stars:>20",                   "Education"),
    ("education platform fullstack stars:>30",               "Education"),
    # Finance
    ("topic:fintech stars:>50",                              "Finance"),
    ("personal finance tracker stars:>30",                   "Finance"),
    ("topic:banking stars:>30",                              "Finance"),
    ("topic:cryptocurrency stars:>50",                       "Finance"),
    ("budget tracker app stars:>30",                         "Finance"),
    ("topic:payment-gateway stars:>30",                      "Finance"),
    ("investment portfolio tracker stars:>30",               "Finance"),
    ("loan management system stars:>20",                     "Finance"),
    ("accounting software stars:>30",                        "Finance"),
    ("topic:expense-tracker stars:>30",                      "Finance"),
    # Social Media
    ("topic:social-network stars:>50",                       "Social Media"),
    ("social media platform fullstack stars:>100",           "Social Media"),
    ("topic:chat-app stars:>50",                             "Social Media"),
    ("topic:messaging-app stars:>40",                        "Social Media"),
    ("real time chat application stars:>50",                 "Social Media"),
    ("topic:twitter-clone stars:>30",                        "Social Media"),
    ("topic:instagram-clone stars:>30",                      "Social Media"),
    ("community forum platform stars:>30",                   "Social Media"),
    ("topic:social-app stars:>30",                           "Social Media"),
    # Logistics
    ("topic:logistics stars:>30",                            "Logistics"),
    ("fleet management system stars:>20",                    "Logistics"),
    ("delivery tracking app stars:>30",                      "Logistics"),
    ("warehouse management system stars:>20",                "Logistics"),
    ("supply chain management stars:>20",                    "Logistics"),
    ("topic:tracking-system stars:>20",                      "Logistics"),
    ("route optimization system stars:>20",                  "Logistics"),
    ("inventory management system stars:>30",                "Logistics"),
    ("shipping tracking system stars:>20",                   "Logistics"),
    # Real Estate
    ("topic:real-estate stars:>30",                          "Real Estate"),
    ("property listing platform stars:>20",                  "Real Estate"),
    ("real estate management system stars:>20",              "Real Estate"),
    ("topic:property-management stars:>20",                  "Real Estate"),
    ("rental management system stars:>20",                   "Real Estate"),
    ("real estate crm stars:>15",                            "Real Estate"),
    ("property booking system stars:>15",                    "Real Estate"),
    ("house rental platform stars:>20",                      "Real Estate"),
    ("real estate website fullstack stars:>20",              "Real Estate"),
    # HR & Recruitment
    ("topic:hrms stars:>20",                                 "HR & Recruitment"),
    ("applicant tracking system stars:>30",                  "HR & Recruitment"),
    ("topic:payroll stars:>20",                              "HR & Recruitment"),
    ("human resource management stars:>20",                  "HR & Recruitment"),
    ("employee management system stars:>20",                 "HR & Recruitment"),
    ("topic:recruitment stars:>20",                          "HR & Recruitment"),
    ("leave management system stars:>15",                    "HR & Recruitment"),
    ("performance review system stars:>15",                  "HR & Recruitment"),
    ("job portal fullstack stars:>20",                       "HR & Recruitment"),
    # IoT & Smart Systems
    ("topic:iot-dashboard stars:>30",                        "IoT & Smart Systems"),
    ("smart home automation stars:>30",                      "IoT & Smart Systems"),
    ("topic:iot stars:>50 language:python",                  "IoT & Smart Systems"),
    ("topic:smart-home stars:>30",                           "IoT & Smart Systems"),
    ("iot monitoring system stars:>20",                      "IoT & Smart Systems"),
    ("topic:home-automation stars:>30",                      "IoT & Smart Systems"),
    ("sensor data dashboard stars:>20",                      "IoT & Smart Systems"),
    ("topic:iot-platform stars:>20",                         "IoT & Smart Systems"),
    # Travel & Hospitality
    ("hotel booking system stars:>30",                       "Travel & Hospitality"),
    ("travel app fullstack stars:>30",                       "Travel & Hospitality"),
    ("topic:hotel-management stars:>20",                     "Travel & Hospitality"),
    ("flight booking system stars:>20",                      "Travel & Hospitality"),
    ("restaurant management system stars:>20",               "Travel & Hospitality"),
    ("topic:booking-system stars:>30",                       "Travel & Hospitality"),
    ("travel booking platform stars:>20",                    "Travel & Hospitality"),
    ("tour management system stars:>15",                     "Travel & Hospitality"),
]

In [5]:
# ── KEYWORD DETECTION MAPS ─────────────────────────────────────────────────────
FRONTEND_KEYWORDS = {
    "React":        ["react", "reactjs", "react.js", "jsx", "create-react-app",
                     "redux", "react-router", "react-query", "react-hooks"],
    "Next.js":      ["next.js", "nextjs", "next js"],
    "Vue.js":       ["vue", "vuejs", "vue.js", "nuxtjs", "nuxt", "vuex"],
    "Angular":      ["angular", "angularjs", "angular cli", "rxjs", "ngrx", "@angular"],
    "Svelte":       ["svelte", "sveltekit"],
    "jQuery":       ["jquery"],
    "Flutter":      ["flutter", "dart"],
    "React Native": ["react native", "expo", "react-native"],
    "Bootstrap":    ["bootstrap", "tailwind"],
    "HTML/CSS":     ["html5", "css3", "static site"],
}
BACKEND_KEYWORDS = {
    "Node.js":     ["node.js", "nodejs", "express", "expressjs",
                    "nestjs", "fastify", "koa", "socket.io"],
    "Django":      ["django", "django rest", "drf", "django-rest-framework"],
    "Flask":       ["flask", "flask-restful"],
    "FastAPI":     ["fastapi", "uvicorn", "starlette"],
    "Laravel":     ["laravel", "artisan", "eloquent"],
    "Spring Boot": ["spring boot", "springboot", "spring-boot", "spring framework"],
    "Rails":       ["ruby on rails", "rails", "ror"],
    "Go":          ["golang", "gin", "go fiber", "echo framework"],
    "ASP.NET":     ["asp.net", ".net core", "dotnet", "blazor"],
    "PHP":         ["codeigniter", "symfony"],
}
DATABASE_KEYWORDS = {
    "MongoDB":       ["mongodb", "mongoose", "mongo db", "atlas"],
    "PostgreSQL":    ["postgresql", "postgres", "psql", "supabase", "neon"],
    "MySQL":         ["mysql", "mariadb"],
    "SQLite":        ["sqlite"],
    "Firebase":      ["firebase", "firestore", "realtime database"],
    "Redis":         ["redis"],
    "DynamoDB":      ["dynamodb"],
    "Cassandra":     ["cassandra"],
    "Elasticsearch": ["elasticsearch", "opensearch"],
    "SQL Server":    ["sql server", "mssql"],
    "Oracle":        ["oracle db", "oracle database"],
}
LANGUAGE_TO_BACKEND  = {
    "JavaScript":"Node.js", "TypeScript":"Node.js", "Python":"Django",
    "PHP":"Laravel",        "Java":"Spring Boot",   "Kotlin":"Spring Boot",
    "Ruby":"Rails",         "Go":"Go",              "C#":"ASP.NET", "Dart":"Flutter",
}
LANGUAGE_TO_FRONTEND = {
    "JavaScript":"React", "TypeScript":"React", "Dart":"Flutter",
}
DOMAIN_DEFAULTS = {
    "Ecommerce":           ("React",   "Node.js",     "MongoDB"),
    "Healthcare":          ("React",   "Django",      "PostgreSQL"),
    "Education":           ("React",   "Node.js",     "MongoDB"),
    "Finance":             ("Angular", "Spring Boot", "PostgreSQL"),
    "Social Media":        ("React",   "Node.js",     "MongoDB"),
    "Logistics":           ("React",   "Node.js",     "PostgreSQL"),
    "Real Estate":         ("React",   "Django",      "PostgreSQL"),
    "HR & Recruitment":    ("Angular", "Spring Boot", "PostgreSQL"),
    "IoT & Smart Systems": ("React",   "Node.js",     "MongoDB"),
    "Travel & Hospitality":("React",   "Node.js",     "PostgreSQL"),
}
DOMAIN_NFRS = {
    "Ecommerce":           ["scalability","security","high availability","performance",
                            "SEO optimization","mobile responsiveness","PCI-DSS compliance"],
    "Healthcare":          ["HIPAA compliance","data privacy","high availability",
                            "security","audit logging","reliability","data encryption"],
    "Education":           ["scalability","performance","accessibility",
                            "mobile responsiveness","real-time support","WCAG compliance"],
    "Finance":             ["security","PCI-DSS compliance","high availability",
                            "audit trail","performance","encryption","regulatory compliance"],
    "Social Media":        ["scalability","real-time performance","low latency",
                            "CDN integration","high availability","media optimization"],
    "Logistics":           ["real-time updates","GPS integration","reliability",
                            "scalability","offline support","performance"],
    "Real Estate":         ["SEO","performance","mobile responsiveness",
                            "security","scalability","fast image loading"],
    "HR & Recruitment":    ["data privacy","security","scalability",
                            "compliance","role-based access","audit logging"],
    "IoT & Smart Systems": ["real-time processing","low latency","reliability",
                            "security","scalability","interoperability"],
    "Travel & Hospitality":["performance","scalability","high availability",
                            "SEO","mobile responsiveness","multi-language support"],
}
SIZE_CONFIG = {
    "Small":  {"team":(1,4),   "budget":"Low",    "duration":(1,5)},
    "Medium": {"team":(4,12),  "budget":"Medium", "duration":(4,12)},
    "Large":  {"team":(12,40), "budget":"High",   "duration":(8,24)},
}

In [6]:
# ── HELPERS ────────────────────────────────────────────────────────────────────
def detect_tech(text, keyword_map):
    tl = text.lower()
    for tech, kws in keyword_map.items():
        if any(kw in tl for kw in kws):
            return tech
    return "Unknown"

def get_readme(owner, repo):
    try:
        r = requests.get(f"https://api.github.com/repos/{owner}/{repo}/readme",
                         headers=HEADERS, timeout=10)
        if r.status_code == 200:
            return base64.b64decode(r.json().get("content","")).decode("utf-8",errors="ignore")
    except Exception:
        pass
    return ""

def get_languages(owner, repo):
    try:
        r = requests.get(f"https://api.github.com/repos/{owner}/{repo}/languages",
                         headers=HEADERS, timeout=10)
        if r.status_code == 200:
            return r.json()
    except Exception:
        pass
    return {}

def classify_size(stars, forks):
    score = stars + forks * 2
    if score < 100:   return "Small"
    elif score < 600: return "Medium"
    else:             return "Large"

def fill_columns(domain, size_label):
    nfrs = DOMAIN_NFRS.get(domain, ["scalability","security","performance"])
    nfr  = ", ".join(random.sample(nfrs, min(3, len(nfrs))))
    cfg  = SIZE_CONFIG.get(size_label, SIZE_CONFIG["Medium"])
    return nfr, random.randint(*cfg["team"]), cfg["budget"], random.randint(*cfg["duration"])

def check_rate_limit():
    try:
        r    = requests.get("https://api.github.com/rate_limit", headers=HEADERS, timeout=10)
        data = r.json()
        rem  = data["resources"]["search"]["remaining"]
        rst  = data["resources"]["search"]["reset"]
        if rem < 3:
            wait = max(rst - time.time() + 5, 0)
            print(f"  ⚠ Rate limit low — waiting {int(wait)}s...")
            time.sleep(wait)
        return rem
    except Exception:
        return 10

In [7]:
# ── PAGINATED SEARCH ────────────────────────────────────────────────────────────
def search_repos(query, domain):
    url       = "https://api.github.com/search/repositories"
    all_items = []

    for page in range(1, MAX_PAGES + 1):
        params = {"q": query, "sort":"stars", "order":"desc",
                  "per_page": 100, "page": page}
        try:
            r = requests.get(url, headers=HEADERS, params=params, timeout=15)
            if r.status_code == 403:
                print("  ⚠ Rate limit — waiting 65s...")
                time.sleep(65)
                r = requests.get(url, headers=HEADERS, params=params, timeout=15)
            if r.status_code != 200:
                print(f"  ✗ HTTP {r.status_code} page {page}")
                break
        except Exception as e:
            print(f"  ✗ {e}")
            break

        items = r.json().get("items", [])
        if not items:
            break
        print(f"    Page {page}: {len(items)} repos")
        all_items.extend(items)
        if len(items) < 100:
            break          # fewer than full page → no more results
        time.sleep(PAGE_DELAY)

    records = []
    for item in all_items:
        owner    = item["owner"]["login"]
        repo     = item["name"]
        stars    = item.get("stargazers_count", 0)
        forks    = item.get("forks_count", 0)
        language = item.get("language") or "Unknown"
        topics   = item.get("topics", [])
        desc     = item.get("description") or ""
        html_url = item.get("html_url", "")

        print(f"    → {owner}/{repo} ({stars}★)")

        readme    = get_readme(owner, repo)
        full_text = f"{desc} {' '.join(topics)} {readme[:4000]}"

        frontend = detect_tech(full_text, FRONTEND_KEYWORDS)
        backend  = detect_tech(full_text, BACKEND_KEYWORDS)
        database = detect_tech(full_text, DATABASE_KEYWORDS)

        # Language API fallback
        if frontend == "Unknown" or backend == "Unknown":
            langs = get_languages(owner, repo)
            time.sleep(0.3)
            if langs:
                primary = max(langs, key=langs.get)
                if backend  == "Unknown" and primary in LANGUAGE_TO_BACKEND:
                    backend  = LANGUAGE_TO_BACKEND[primary]
                if frontend == "Unknown":
                    for lang in langs:
                        if lang in LANGUAGE_TO_FRONTEND:
                            frontend = LANGUAGE_TO_FRONTEND[lang]
                            break

        # Domain default fallback — guarantees zero Unknowns
        def_fe, def_be, def_db = DOMAIN_DEFAULTS.get(domain, ("React","Node.js","MongoDB"))
        if frontend == "Unknown": frontend = def_fe
        if backend  == "Unknown": backend  = def_be
        if database == "Unknown": database = def_db

        cloud_kws  = ["docker","kubernetes","heroku","aws","azure","gcp","vercel","netlify"]
        deployment = "Cloud" if any(k in full_text.lower() for k in cloud_kws) else "On-premise"
        size_label = classify_size(stars, forks)
        nfr, team, budget, duration = fill_columns(domain, size_label)

        records.append({
            "Project_ID":                  "",
            "Project_Name":                repo.replace("-"," ").replace("_"," ").title(),
            "Domain":                      domain,
            "Project_Description":         desc[:250] or "No description",
            "Functional_Requirements":     ", ".join(topics[:6]) if topics else "See README",
            "Non_Functional_Requirements": nfr,
            "Project_Size":                size_label,
            "Team_Size":                   team,
            "Budget_Level":                budget,
            "Duration_Months":             duration,
            "Deployment":                  deployment,
            "Frontend_Tech":               frontend,
            "Backend_Tech":                backend,
            "Database":                    database,
            "Source":                      "GitHub",
            "GitHub_URL":                  html_url,
            "Stars":                       stars,
            "Forks":                       forks,
            "Primary_Language":            language,
        })
        time.sleep(DELAY)

    return records

In [8]:
# ── MAIN ───────────────────────────────────────────────────────────────────────
def main():
    print("=" * 65)
    print("  GitHub Tech Stack Collector — All domains, no row limit")
    print(f"  Queries : {len(SEARCH_QUERIES)} | Pages/query: {MAX_PAGES} | Per page: 100")
    print(f"  Max possible rows: ~{len(SEARCH_QUERIES)*100*MAX_PAGES} (before dedup)")
    print(f"  Started : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 65)

    all_records   = []
    seen_urls     = set()
    domain_counts = Counter()

    for qi, (query, domain) in enumerate(SEARCH_QUERIES, 1):
        print(f"\n[{qi}/{len(SEARCH_QUERIES)}] [{domain}] {query}")

        if qi % 10 == 0:
            rem = check_rate_limit()
            print(f"  API calls remaining: {rem}")

        records = search_repos(query, domain)

        added = 0
        for rec in records:
            if rec["GitHub_URL"] not in seen_urls:
                seen_urls.add(rec["GitHub_URL"])
                all_records.append(rec)
                domain_counts[domain] += 1
                added += 1

        print(f"  ✓ +{added} new  |  Total unique: {len(all_records)}")
        time.sleep(QUERY_DELAY)

    # ── Assign IDs, shuffle, save ──────────────────────────────────────────
    random.shuffle(all_records)
    for i, rec in enumerate(all_records, 1):
        rec["Project_ID"] = f"GH{i:04d}"

    df = pd.DataFrame(all_records)
    df.to_excel(OUTPUT_FILE, index=False, sheet_name="GitHub_Projects")

    # ── Final report ───────────────────────────────────────────────────────
    print(f"\n{'=' * 65}")
    print(f"  ✓ Saved {len(df)} projects → {OUTPUT_FILE}")
    print(f"\n  Domain breakdown:")
    for domain, count in df["Domain"].value_counts().items():
        print(f"    {domain:<28} {count}")
    print(f"\n  Column completeness (all should be 100%):")
    for col in ["Non_Functional_Requirements","Team_Size","Budget_Level",
                "Duration_Months","Frontend_Tech","Backend_Tech","Database"]:
        unknowns = (df[col].astype(str).isin(["Unknown","","nan"])).sum()
        filled   = len(df) - unknowns
        pct      = 100 * filled // len(df)
        print(f"    {col:<35} {filled}/{len(df)} ({pct}%)")
    print(f"\n  Tech distribution (top 5):")
    print(f"    Frontend : {dict(df['Frontend_Tech'].value_counts().head(5))}")
    print(f"    Backend  : {dict(df['Backend_Tech'].value_counts().head(5))}")
    print(f"    Database : {dict(df['Database'].value_counts().head(5))}")
    print("=" * 65)

    try:
        from google.colab import files
        files.download(OUTPUT_FILE)
        print("  ⬇ Download started!")
    except ImportError:
        print(f"  File saved locally: {OUTPUT_FILE}")

if __name__ == "__main__":
    main()

Streaming output truncated to the last 5000 lines.
    → cassidoo/shopify-react-astro (123★)
    → Elkfox/Ajaxinate (121★)
    → jtrackingai/analytics-tracking-automation (120★)
    → ericdude4/shopifex (115★)
    → lucasvocos/gatsby-sanity-shopify (108★)
    → ctrlaltdylan/shopify-session-tokens-nextjs (106★)
    → tokenblast/shopify-spy (105★)
    → internalfx/quickshot (104★)
    → nsweeting/shopify (103★)
    → gil--/gatsby-starter-shopifypwa (101★)
    → RobertBroersma/next-storefront (101★)
    → voxjs/vox (101★)
    → cam/Shopify-Theme-Framework (101★)
    → packdigital/pack-hydrogen-theme-blueprint (99★)
    → Shopify/polaris-telescope (98★)
    → solana-labs/solana-payments-app (96★)
    → culturekings/shopify-json-parser (93★)
    → sanity-io/hydrogen-sanity (92★)
    → ThinkSimple/flutter_simple_shopify (90★)
    → Elkfox/shopify-node-app (88★)
    → donutdan4114/shopify (86★)
    → Miragic-AI/VirtualTryOn (85★)
    → Mini-Sylar/shopify-app-vue-template (85★)
    → Elkfox/sh

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  ⬇ Download started!
